# 02 · Silver de población
**Responsable:** P1 · **Código:** `src/silver_poblacion.py` · **Salidas:** `silver/poblacion.parquet`, `silver/poblacion_area.parquet`, figura de la pirámide

**Qué hace**
- `poblacion`: lee el CSV de bronze como texto, normaliza el código DIVIPOLA a 5 dígitos, convierte tipos y aplica reglas: código válido, año 2020–2025, conteos no nulos ni negativos, 18–44 ≤ 18–64 ≤ total, hombres + mujeres = total y clave única municipio-año.
- `poblacion_area`: el archivo de "centros poblados" son proyecciones por **área** (cabecera, rural y total). Se verifica que cabecera + rural = total y se conservan solo cabecera y rural, en formato largo (sexo × edad). Así nunca se suma dos veces la población.
- Concilia el total de esa fuente con la población DANE oficial y deja la diferencia en `outputs/quality/conciliacion_poblacion_fuentes.csv`.

**Decisión clave:** la población municipal oficial es `poblacion_dane`; el archivo por áreas se usa para la pirámide, el % rural y la sensibilidad.

**Qué revisar al ejecutar:** 738 filas en `poblacion` (123 × 6 años), 0 rechazos esperados, y el tamaño de las diferencias en la conciliación.

In [1]:
import sys, pathlib
ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT))
import polars as pl
from src.config import *
asegurar_carpetas()

In [2]:
from src.silver_poblacion import construir_poblacion, construir_poblacion_area
pob = construir_poblacion()
pob.head()

[silver/poblacion] revisados=738 validos=738 rechazados=0


codigo_municipio,municipio,anio,poblacion_total,hombres,mujeres,pob_18_44,pob_18_64,hombres_18_44,mujeres_18_44
str,str,i16,i64,i64,i64,i64,i64,i64,i64
"""15001""","""Tunja""",2020,178975,84968,94007,78483,115033,37714,40769
"""15001""","""Tunja""",2021,181676,86287,95389,80495,117688,38762,41733
"""15001""","""Tunja""",2022,183818,87296,96522,82109,119907,39607,42502
"""15001""","""Tunja""",2023,185748,88227,97521,83495,121900,40356,43139
"""15001""","""Tunja""",2024,187765,89226,98539,84821,123880,41078,43743


In [3]:
areas = construir_poblacion_area(poblacion=pob)
areas.head()

[silver/centros_poblados] revisados=1845 validos=1845 rechazados=0
[silver/poblacion_area] revisados=211560 validos=211560 rechazados=0


codigo_municipio,anio,area,sexo,edad,personas
str,i16,str,str,i16,i64
"""15001""",2020,"""cabecera""","""H""",0,1060
"""15001""",2020,"""cabecera""","""H""",1,1081
"""15001""",2020,"""cabecera""","""H""",2,1101
"""15001""",2020,"""cabecera""","""H""",3,1126
"""15001""",2020,"""cabecera""","""H""",4,1146


In [4]:
pl.read_csv(QUALITY / "conciliacion_poblacion_fuentes.csv").select(
    pl.col("diferencia").min().alias("min"), pl.col("diferencia").median().alias("mediana"),
    pl.col("diferencia").max().alias("max"))

min,mediana,max
i64,f64,i64
-1679,106.0,2880


In [9]:
from src import graficos
graficos.piramide_poblacional(areas.to_pandas(), ANIO_PIRAMIDE)

PosixPath('/mnt/c/Users/TUPTC/bigdata/big-data-postgraduate/proyecto_boyaca/outputs/figures/p1_piramide_2023.png')

In [7]:
pl.read_csv(QUALITY / "metricas_silver.csv").filter(pl.col("fuente").is_in(["poblacion", "centros_poblados", "poblacion_area"]))

fecha_ejecucion,capa,fuente,regla,descripcion,revisados,afectados,resultado,accion
str,str,str,str,str,i64,i64,str,str
"""2026-11-06T19:31:59""","""silver""","""poblacion""","""codigo_divipola_valido""","""Código de 5 dígitos que inicia…",738,0,"""OK""","""rechazar"""
"""2026-11-06T19:31:59""","""silver""","""poblacion""","""anio_en_rango""","""Año entre 2020 y 2025""",738,0,"""OK""","""rechazar"""
"""2026-11-06T19:31:59""","""silver""","""poblacion""","""conteos_no_nulos""","""Ningún conteo vacío o no numér…",738,0,"""OK""","""rechazar"""
"""2026-11-06T19:31:59""","""silver""","""poblacion""","""conteos_no_negativos""","""Conteos >= 0""",738,0,"""OK""","""rechazar"""
"""2026-11-06T19:31:59""","""silver""","""poblacion""","""total_positivo""","""Población total > 0""",738,0,"""OK""","""rechazar"""
…,…,…,…,…,…,…,…,…
"""2026-11-06T19:32:00""","""silver""","""poblacion_area""","""cardinalidad_anio_2023""","""anio=2023: se esperaban 123 mu…",123,0,"""OK""","""alertar"""
"""2026-11-06T19:32:00""","""silver""","""poblacion_area""","""cardinalidad_anio_2020""","""anio=2020: se esperaban 123 mu…",123,0,"""OK""","""alertar"""
"""2026-11-06T19:32:00""","""silver""","""poblacion_area""","""cardinalidad_anio_2021""","""anio=2021: se esperaban 123 mu…",123,0,"""OK""","""alertar"""
